# 8장 분포 읽기: 평균 너머의 데이터

『AI 시대의 파이썬 데이터 시각화』 예제 노트북입니다. 셀 첫 줄의 번호(P·C·B)는 책 지면의 코드 번호와 같습니다. 위에서부터 차례로 실행합니다.

## 준비

새 노트북이면 이 장의 코드보다 먼저 아래 준비 셀을 실행합니다(책의 앞 장과 같은 셀).

**코랩이면 이 셀(P02)을, 내 컴퓨터면 다음 셀(P03)을 실행합니다.** 둘 다 실행하지 않습니다.

In [ ]:
# P02 | 한글 글꼴(코랩): 나눔글꼴을 설치하고 Matplotlib에 등록합니다.
!apt-get -qq update
!apt-get -qq install -y fonts-nanum

import matplotlib.pyplot as plt
from matplotlib import font_manager

for path in font_manager.findSystemFonts():
    if 'Nanum' in path:
        font_manager.fontManager.addfont(path)
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

**내 컴퓨터(Mac·Windows)면 이 셀을 실행합니다.** Windows는 글꼴 이름을 `'Malgun Gothic'`으로 바꿉니다.

In [ ]:
# P03 | 한글 글꼴(Mac·Windows): 그래프를 그리기 전에 한 번 실행합니다.
import matplotlib.pyplot as plt

plt.rcParams['font.family'] = 'AppleGothic'   # Windows는 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False    # 음수 기호가 깨지지 않게

## 8.1 평균이 숨기는 차이

In [ ]:
# C01 | 배송 표: 한 행은 배송 한 건이고, 두 지점이 20건씩입니다.
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

a_minutes = [18, 19, 19, 20, 20, 20, 21, 21, 22, 20] * 2   # A지점(분)
b_minutes = [8, 10, 12, 14, 16, 18, 20, 22, 25, 55] * 2    # B지점(분)
delivery = pd.DataFrame({'branch': ['A'] * 20 + ['B'] * 20,
                         'minutes': a_minutes + b_minutes})
delivery['delivery_id'] = [f'D{i:02}' for i in range(1, 41)]
branch_colors = {'A': '#0072B2', 'B': '#D55E00'}   # 지점 색(4장과 같음)
print(delivery.head(3))
print(delivery.groupby('branch').size())

In [ ]:
# C02 | 평균과 오차 표시: 평균만 그린 막대에 오차 표시 두 가지를 더합니다.
fig, axes = plt.subplots(1, 3, figsize=(6, 2.2), layout='constrained',
                         sharey=True)
sns.barplot(data=delivery, x='branch', y='minutes', hue='branch',
            palette=branch_colors, legend=False, errorbar=None,
            ax=axes[0])
sns.barplot(data=delivery, x='branch', y='minutes', hue='branch',
            palette=branch_colors, legend=False, capsize=0.2, seed=42,
            ax=axes[1])                  # errorbar를 빼면 ('ci', 95)
sns.barplot(data=delivery, x='branch', y='minutes', hue='branch',
            palette=branch_colors, legend=False, errorbar=('pi', 100),
            capsize=0.2, ax=axes[2])     # 가운데 100% = 최솟값~최댓값
axes[0].set(title='평균만', xlabel='지점', ylabel='배송시간 (분)')
axes[1].set(title='기본값: 95% 신뢰구간', xlabel='지점')
axes[2].set(title='최솟값~최댓값', xlabel='지점')
plt.show()

In [ ]:
# C03 | 평균과 중앙값: 요약값 여럿을 이름을 붙여 한 표로 봅니다.
summary = delivery.groupby('branch')['minutes'].agg(
    count='count', mean='mean', median='median', min='min', max='max')
print(summary)

## 8.2 히스토그램의 구간

In [ ]:
# C04 | 히스토그램: 같은 구간 경계로 지점마다 배송 건수를 셉니다.
from matplotlib.ticker import MaxNLocator
edges = np.arange(0, 61, 5)                  # 0, 5, …, 60분
fig, axes = plt.subplots(1, 2, figsize=(6, 2.4), layout='constrained',
                         sharex=True, sharey=True)
for ax, (branch, part) in zip(axes, delivery.groupby('branch')):
    sns.histplot(data=part, x='minutes', bins=edges,
                 color=branch_colors[branch], ax=ax)
    ax.set(title=f'{branch}지점 (n={len(part)})', xlabel='배송시간 (분)',
           xlim=(0, 60), xticks=[0, 20, 40, 60])
    ax.yaxis.set_major_locator(MaxNLocator(integer=True))   # 정수 눈금
axes[0].set_ylabel('건수')
plt.show()

In [ ]:
# C04B | 구간 검산: 구간별 건수를 숫자로 꺼내 합을 확인합니다.
b_rows = delivery[delivery['branch'] == 'B']
b_values = b_rows['minutes']
counts, bin_edges = np.histogram(b_values, bins=edges)
table = pd.DataFrame({'left': bin_edges[:-1], 'right': bin_edges[1:],
                      'count': counts})
print(table[table['count'] > 0].to_string(index=False))
print('건수 합:', counts.sum())
probe, _ = np.histogram([20, 60], bins=edges)    # 경계에 걸린 두 값
print('두 값이 든 구간의 왼쪽 경계:', edges[:-1][probe > 0])

In [ ]:
# C05 | 구간 폭: 같은 B지점 20건을 폭 2·5·10분으로 나눠 봅니다.
fig, axes = plt.subplots(1, 3, figsize=(6, 2.2), layout='constrained',
                         sharex=True)
for ax, width in zip(axes, [2, 5, 10]):
    sns.histplot(data=b_rows, x='minutes', bins=np.arange(0, 61, width),
                 color=branch_colors['B'], ax=ax)
    ax.set(title=f'폭 {width}분', xlabel='배송시간 (분)', ylabel='건수',
           xlim=(0, 60))
    ax.yaxis.set_major_locator(MaxNLocator(integer=True))
plt.show()

### 직접 해 보기

**W8-01** B지점 히스토그램의 구간 폭을 고릅니다.

1. C05처럼 B지점 히스토그램을 폭 4분과 10분으로 그리고, 두 그림 모두 X축을 0~60분으로 고정합니다.
2. C04B처럼 `np.histogram()`으로 두 폭의 건수 합이 관측 수와 같은지, 55분이 들어갔는지 확인합니다.
3. 두 폭에서 같게 보이는 점 하나와 폭에 따라 달라지는 모양을 나눠 적습니다.
4. 보고서에 쓸 폭을 고르고 이유를 한 문장으로 씁니다.

**완료 기준**: 두 폭의 건수 합이 관측 수와 같고 55분이 들어 있으며, 공통 관찰과 폭에 따른 차이를 구분했고, 고른 폭의 이유가 있다.

In [ ]:
# W8-01 — 여기에서 직접 해 봅니다.

## 8.3 밀도로 집단 비교

In [ ]:
# C06 | 밀도: 지점마다 넓이를 1로 맞춰 관측 수와 상관없이 모양을 봅니다.
fig, ax = plt.subplots(figsize=(6, 2.6), layout='constrained')
for branch, part in delivery.groupby('branch'):
    sns.histplot(data=part, x='minutes', bins=edges, stat='density',
                 element='step', fill=False, color=branch_colors[branch],
                 linestyle='-' if branch == 'A' else '--', label=branch,
                 ax=ax)
ax.fill_between([15, 20], 4 / (20 * 5), color=branch_colors['B'],
                alpha=0.25)              # 높이 = 4건 ÷ (20건 × 5분)
ax.legend(title='지점')
ax.set(xlabel='배송시간 (분)', ylabel='밀도 (1/분)', xlim=(0, 60),
       yticks=[0, 0.04, 0.08, 0.12])
plt.show()

In [ ]:
# C06B | 넓이 검산: 밀도 높이에 구간 폭을 곱하면 구간의 비율입니다.
density, bin_edges = np.histogram(b_values, bins=edges, density=True)
areas = density * 5                          # 높이 × 폭 5분
print('B [15,20) 높이:', density[3], '(1/분)')
print('B [15,20) 넓이:', round(areas[3], 2), '= 20건 중 4건')
print('넓이의 합:', round(areas.sum(), 2))

### 직접 해 보기

**W8-02** 관측 수가 다를 때 건수와 밀도를 견줍니다.

1. `pd.concat()`(4장 C07)으로 B지점 행을 한 번 더 이어 붙여 A 20건·B 40건인 비교용 표를 만듭니다. 이어 붙인 행은 `delivery_id`가 되풀이되므로 새로 관측한 배송이 아닙니다.
2. 그리기 전에 예측합니다. 건수 그림과 밀도 그림에서 B지점은 각각 어떻게 달라질까요?
3. `hue='branch'`로 왼쪽은 `stat='count'`, 오른쪽은 `stat='density'`와 `common_norm=False`로 그리고, 제목에 복제한 교육용 표임을 밝힙니다.
4. 예측과 결과를 견주고, 밀도만 보고 관측 수를 말할 수 없는 이유를 한 문장으로 씁니다. 오른쪽을 `common_norm=False` 없이도 그려 지점별 넓이 1과 두 지점을 합친 넓이 1을 견줍니다.

**완료 기준**: 건수·밀도에서 B지점의 변화를 예측과 견주어 적었고, 복제한 표임을 밝혔으며, 두 정규화의 차이와 복제가 새 관측이 아님을 설명했다.

In [ ]:
# W8-02 — 여기에서 직접 해 봅니다.

## 8.4 박스 플롯과 이상치 후보

In [ ]:
# C07 | 박스 플롯: 중앙값과 가운데 50% 위에 원자료 점을 찍습니다.
fig, ax = plt.subplots(figsize=(6, 2.6), layout='constrained')
sns.boxplot(data=delivery, x='branch', y='minutes', hue='branch',
            palette=branch_colors, legend=False, width=0.45,
            showfliers=False, ax=ax)     # 수염 밖 점은 원자료로 대신
np.random.seed(9)                        # 점의 가로 흔들림을 재현
sns.stripplot(data=delivery, x='branch', y='minutes', color='#333333',
              jitter=0.12, alpha=0.55, size=4, ax=ax)
ax.set(xlabel='지점', ylabel='배송시간 (분)')
plt.show()

In [ ]:
# C08 | 이상치 후보: 사분위수로 경계를 계산하고 경계 밖 행을 찾습니다.
quart = delivery.groupby('branch')['minutes'].quantile([0.25, 0.5, 0.75])
q = quart.unstack()                          # 행 = 지점, 열 = 분위
q.columns = ['Q1', 'median', 'Q3']
q['IQR'] = q['Q3'] - q['Q1']
q['lower'] = q['Q1'] - 1.5 * q['IQR']        # 아래 경계
q['upper'] = q['Q3'] + 1.5 * q['IQR']        # 위 경계
checked = delivery.join(q[['lower', 'upper']], on='branch')
outside = ((checked['minutes'] < checked['lower'])
           | (checked['minutes'] > checked['upper']))
print(q)
print(checked.loc[outside, ['delivery_id', 'branch', 'minutes']])

In [ ]:
# C08B | 경계와 수염: 수염은 경계 안 가장 바깥 관측까지입니다.
print(pd.Series([10, 20, 40, 80]).quantile([0.25, 0.5, 0.75]).tolist())
inside = checked['minutes'].between(checked['lower'], checked['upper'])
b_inside = checked.loc[inside & (checked['branch'] == 'B'), 'minutes']
print('B 수염:', b_inside.min(), '~', b_inside.max(), '분')
fig, ax = plt.subplots(figsize=(6, 2.0), layout='constrained')
sns.boxplot(x=b_values, color=branch_colors['B'], width=0.4, ax=ax)
for fence in q.loc['B', ['lower', 'upper']]:     # 아래·위 경계
    ax.axvline(fence, color='#666666', linestyle='--', linewidth=1)
ax.annotate('55분 두 건(겹침)', xy=(55, 0), xytext=(42, -0.32),
            arrowprops={'arrowstyle': '->', 'color': '#333333'})
ax.set(xlim=(-8, 62), yticks=[], xlabel='배송시간 (분)',
       title='B지점: 경계는 -3분과 37분, 수염은 8분과 25분')
plt.show()

### 직접 해 보기

**W8-03** A지점으로 경계와 수염을 구분합니다.

1. C08의 출력에서 A지점의 Q1·Q3·IQR·경계를 읽습니다.
2. C08B처럼 A지점의 아래·위 수염을 구하고 경계와 어떻게 다른지 한 문장으로 적습니다.
3. "A지점의 위 수염은 위 경계까지 올라간다"를 바른 문장으로 고칩니다.
4. 55분 두 건을 지우기 전에 확인할 기록을 적고 `len(delivery)`로 원본 행 수를 확인합니다.

**완료 기준**: A지점의 경계와 수염을 구분해 적었고, 틀린 문장을 경계와 수염의 뜻으로 고쳤으며, 후보의 확인 기록이 있고 원본 행을 지우지 않았다.

In [ ]:
# W8-03 — 여기에서 직접 해 봅니다.

## 8.5 차트 선택과 분포 리포트

In [ ]:
# C09 | 매끄러운 밀도: 밀도 곡선과 바이올린으로 모양을 봅니다.
fig, axes = plt.subplots(1, 2, figsize=(6, 2.6), layout='constrained')
sns.histplot(data=b_rows, x='minutes', bins=edges, stat='density',
             color=branch_colors['B'], alpha=0.3, ax=axes[0])
sns.kdeplot(data=b_rows, x='minutes', cut=0, color=branch_colors['B'],
            ax=axes[0])                  # 같은 20건으로 추정한 밀도 곡선
axes[0].set(title='B지점: 막대와 밀도 곡선', xlabel='배송시간 (분)',
            ylabel='밀도 (1/분)')
sns.violinplot(data=delivery, x='branch', y='minutes', hue='branch',
               palette=branch_colors, legend=False, inner='quart', cut=0,
               ax=axes[1])
axes[1].set(title='바이올린', xlabel='지점', ylabel='배송시간 (분)')
plt.show()

In [ ]:
# C10 | 결측·범위 점검: 확인할 행을 표시하고 검사 대상과 나눕니다.
extra = pd.DataFrame({'branch': ['A', 'B'], 'minutes': [np.nan, -3],
                      'delivery_id': ['CHECK01', 'CHECK02']})
raw = pd.concat([delivery, extra], ignore_index=True)     # 예시 42행
invalid = raw['minutes'].isna() | (raw['minutes'] < 0)
print('전체 행:', len(raw), '| 확인 필요:', invalid.sum())
print(raw[invalid])
valid = raw[~invalid]
print('검사 대상:', len(valid), '행 | 55분:',
      (valid['minutes'] == 55).sum(), '건')

In [ ]:
# C11 | 분포 리포트: 왼쪽은 모양, 오른쪽은 중앙값과 산포를 봅니다.
fig, axes = plt.subplots(1, 2, figsize=(6, 2.6), layout='constrained')
for branch, part in delivery.groupby('branch'):
    sns.histplot(data=part, x='minutes', bins=edges, stat='density',
                 element='step', fill=False, color=branch_colors[branch],
                 linestyle='-' if branch == 'A' else '--', label=branch,
                 ax=axes[0])
axes[0].legend(title='지점')
axes[0].set(title='모양(지점별 넓이 1)', xlabel='배송시간 (분)',
            ylabel='밀도 (1/분)', yticks=[0, 0.04, 0.08, 0.12])
sns.boxplot(data=delivery, x='branch', y='minutes', hue='branch',
            palette=branch_colors, legend=False, ax=axes[1])
axes[1].set(title='중앙값과 산포', xlabel='지점', ylabel='배송시간 (분)',
            ylim=(0, 70))                     # 위쪽에 출처 줄 자리
axes[1].text(0.03, 0.96, '자료: 교육용 가상 자료(배송 40건)',
             transform=axes[1].transAxes, va='top', fontsize=9,
             color='#666666')                        # 출처 줄(6장)
axes[1].axhline(delivery['minutes'].mean(), color='#666666',
                linestyle=':', label='평균(두 지점 모두)')
axes[1].legend(loc='center right', frameon=False)
fig.suptitle('평균은 모두 20분, 분포는 다르다 (지점별 20건)', fontsize=13)
plt.show()

### 직접 해 보기

**W8-04** C11을 분포 리포트로 완성합니다.

1. 오른쪽 박스 플롯에 C07처럼 원자료 점을 더하고 `showfliers=False`로 겹침을 없앱니다.
2. 전체 제목과 축 라벨에서 관측 수·단위·정규화 기준을 확인합니다.
3. `ch08_output/W8-04.png`로 저장하고 다시 열어(2장 C05·C06) 제목·단위·범례가 잘리지 않았는지 봅니다.
4. 위 표의 네 구분에 맞춰 평균·중앙값·범위를 숫자와 단위로 넣은 네 문장을 씁니다.

**완료 기준**: 관측 수·단위·정규화 기준이 그림에 있고, B지점의 수염과 최댓값을 혼동하지 않았으며, 원인을 단정하지 않았고, 저장본을 확인했다.

In [ ]:
# W8-04 — 여기에서 직접 해 봅니다.

## 8.6 해석과 점검

## 직접 해 보기 풀이

먼저 위의 연습 칸에서 직접 해 본 뒤에 봅니다. 풀이는 한 가지 예이고, 완료 기준을 만족하면 다른 코드도 됩니다.

### W8-01

In [ ]:
# W8-01 | 풀이: B지점 히스토그램을 폭 4분과 10분으로 그려 견줍니다.
fig, axes = plt.subplots(1, 2, figsize=(6, 2.4), layout='constrained', sharex=True)
for ax, width in zip(axes, [4, 10]):
    sns.histplot(data=b_rows, x='minutes', bins=np.arange(0, 61, width),
                 color=branch_colors['B'], ax=ax)
    ax.set(title=f'폭 {width}분', xlabel='배송시간 (분)', ylabel='건수', xlim=(0, 60))
    ax.yaxis.set_major_locator(MaxNLocator(integer=True))
plt.show()

In [ ]:
# 2. 두 폭의 건수 합과 55분
for width in [4, 10]:
    counts, bin_edges = np.histogram(b_values, bins=np.arange(0, 61, width))
    at_55 = counts[(bin_edges[:-1] <= 55) & (55 < bin_edges[1:])][0]
    print(f'폭 {width}분: 건수 합 {counts.sum()}, 55분이 든 구간의 건수 {at_55}')
    assert counts.sum() == len(b_values) and at_55 == 2
# 3.(예) 두 폭 모두 55분 두 건이 다른 배송과 멀리 떨어져 있다. 폭 4분은 8~28분에 고르게 퍼진 모양이고,
#   폭 10분은 10~20분 구간 하나에 10건이 몰린 봉우리로 보인다.
# 4.(예) 20건뿐인 자료에서 폭 10분은 퍼진 모양을 한 봉우리로 뭉개므로, 보고서에는 폭 4분을 쓴다.

### W8-02

In [ ]:
# W8-02 | 풀이: B지점 행을 한 번 더 이어 붙인 비교용 표로 건수와 밀도를 견줍니다.
doubled = pd.concat([delivery, delivery[delivery['branch'] == 'B']], ignore_index=True)
print(doubled.groupby('branch').size())
print('되풀이된 delivery_id:', doubled['delivery_id'].duplicated().sum(), '건(새 관측이 아님)')
# 2. 예측(예): 건수 그림에서 B지점 막대는 두 배로 높아지고, 밀도 그림에서는 모양이 그대로일 것이다.
fig, axes = plt.subplots(1, 2, figsize=(6, 2.6), layout='constrained', sharex=True)
sns.histplot(data=doubled, x='minutes', hue='branch', bins=edges, stat='count',
             element='step', palette=branch_colors, ax=axes[0])
sns.histplot(data=doubled, x='minutes', hue='branch', bins=edges, stat='density',
             common_norm=False, element='step', palette=branch_colors, ax=axes[1])
axes[0].set(title='건수', xlabel='배송시간 (분)', ylabel='건수')
axes[1].set(title='밀도(지점별 넓이 1)', xlabel='배송시간 (분)', ylabel='밀도 (1/분)')
for ax in axes:
    ax.get_legend().set_title('지점')
fig.suptitle('복제한 교육용 표: A 20건, B 40건(B를 두 번 이어 붙임)')
plt.show()

In [ ]:
# 결과 확인: B지점의 밀도는 복제 전과 같다
before, _ = np.histogram(b_values, bins=edges, density=True)
after, _ = np.histogram(doubled.loc[doubled['branch'] == 'B', 'minutes'], bins=edges, density=True)
print('B 밀도가 같은가?', np.allclose(before, after))
assert np.allclose(before, after)
# 4. common_norm 기본값(True)이면 두 지점을 합친 넓이가 1 — 지점 넓이는 행 수 비율
fig, ax = plt.subplots(figsize=(6, 2.2), layout='constrained')
sns.histplot(data=doubled, x='minutes', hue='branch', bins=edges, stat='density',
             element='step', palette=branch_colors, ax=ax)
ax.set(title='밀도(두 지점 합친 넓이 1)', xlabel='배송시간 (분)', ylabel='밀도 (1/분)')
plt.show()
print('지점별 넓이:', (doubled.groupby('branch').size() / len(doubled)).round(3).to_dict())
# 4.(예) 예측대로 건수에서는 B지점 막대가 두 배가 됐지만 밀도는 그대로다. 밀도는 지점마다 넓이를 1로 맞춰
#   관측 수를 지우므로, 밀도만 보고는 B지점이 20건인지 40건인지 말할 수 없다.

### W8-03

In [ ]:
# W8-03 | 풀이: A지점의 경계와 수염을 구분합니다.
print(q.loc['A'])                                    # 1. Q1·Q3·IQR·경계
inside = checked['minutes'].between(checked['lower'], checked['upper'])   # C08B와 같은 조건
a_inside = checked.loc[inside & (checked['branch'] == 'A'), 'minutes']
print('A 수염:', a_inside.min(), '~', a_inside.max(), '분')   # 2.
assert (q.loc['A', 'lower'], q.loc['A', 'upper']) == (16, 24) and (a_inside.min(), a_inside.max()) == (18, 22)
# 2.(예) A지점의 경계는 16분과 24분이지만, 수염은 경계 안 가장 바깥 관측인 18분과 22분까지다.
# 3. 고친 문장(예): A지점의 위 수염은 위 경계(24분)가 아니라, 경계 안에서 가장 긴 관측인 22분까지 올라간다.

In [ ]:
# 4. 55분 두 건은 지우기 전에 확인할 기록이 있다(원본 행은 그대로)
print(checked.loc[outside, ['delivery_id', 'branch', 'minutes']])
print('원본 행 수:', len(delivery))
assert len(delivery) == 40
# 확인할 기록(예): 두 건의 주문·출발·도착 시각, 배송 주소와 거리, 그날의 날씨나 교통 기록.

### W8-04

In [ ]:
# W8-04 | 풀이: C11을 분포 리포트로 완성합니다.
from pathlib import Path
from IPython.display import Image, display

output_dir = Path('ch08_output')
output_dir.mkdir(exist_ok=True)
fig, axes = plt.subplots(1, 2, figsize=(6, 2.6), layout='constrained')
for branch, part in delivery.groupby('branch'):
    sns.histplot(data=part, x='minutes', bins=edges, stat='density',
                 element='step', fill=False, color=branch_colors[branch],
                 linestyle='-' if branch == 'A' else '--', label=branch,
                 ax=axes[0])
axes[0].legend(title='지점')
axes[0].set(title='모양(지점별 넓이 1)', xlabel='배송시간 (분)',
            ylabel='밀도 (1/분)', yticks=[0, 0.04, 0.08, 0.12])
sns.boxplot(data=delivery, x='branch', y='minutes', hue='branch',
            palette=branch_colors, legend=False, width=0.45,
            showfliers=False, ax=axes[1])     # 1. 수염 밖 점은 원자료로 대신
np.random.seed(9)
sns.stripplot(data=delivery, x='branch', y='minutes', color='#333333',
              jitter=0.12, alpha=0.55, size=4, ax=axes[1])   # C07처럼 원자료 점
axes[1].set(title='중앙값과 산포', xlabel='지점', ylabel='배송시간 (분)',
            ylim=(0, 70))                     # 위쪽에 출처 줄 자리
axes[1].text(0.03, 0.96, '자료: 교육용 가상 자료(배송 40건)',
             transform=axes[1].transAxes, va='top', fontsize=9,
             color='#666666')                        # 출처 줄(6장)
axes[1].axhline(delivery['minutes'].mean(), color='#666666',
                linestyle=':', label='평균(두 지점 모두)')
axes[1].legend(loc='center right', frameon=False)
fig.suptitle('평균은 모두 20분, 분포는 다르다 (지점별 20건)', fontsize=13)
fig.savefig(output_dir / 'W8-04.png', dpi=150, bbox_inches='tight')   # 3.
plt.show()
display(Image(filename='ch08_output/W8-04.png', width=600))

In [ ]:
# 4. 문장에 쓸 값
print(delivery.groupby('branch')['minutes'].agg(['mean', 'median', 'min', 'max']))
inside = checked['minutes'].between(checked['lower'], checked['upper'])
b_whisker = checked.loc[inside & (checked['branch'] == 'B'), 'minutes']
print('B 수염:', b_whisker.min(), '~', b_whisker.max(), '분')
# 관찰(예): 두 지점의 평균 배송시간은 모두 20분이다.
# 비교(예): 중앙값은 A지점 20분, B지점 17분이고, 범위는 A지점 18~22분, B지점 8~55분이다(B지점의 수염은 8~25분).
# 한계(예): 지점별 20건의 교육용 가상 자료라 실제 배송 전체로 넓혀 말할 수 없다.
# 추가 확인(예): B지점 55분 두 건의 원본 기록을 확인하고, 더 많은 기간의 배송을 모아 다시 본다.